In [11]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [12]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [13]:
def quantum_random_bits(n):
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)           # Hadamard: |0> -> |+>
        qc.measure(i, i)  # Collapse to 0 or 1

    simulator = BasicSimulator()
    job       = simulator.run(transpile(qc, simulator), shots=1)
    counts    = job.result().get_counts()

    # counts has one key (one shot); it's a bit-string with MSB first, so reverse it
    bitstring = list(counts.keys())[0]
    return [int(b) for b in reversed(bitstring)]

In [14]:
# ── ALICE ──────────────────────────────────────────────────────────────────
N = 20

alice_bits  = quantum_random_bits(N)  # Secret bit string Alice wants to share
alice_bases = quantum_random_bits(N)  # Alice's random basis choices

print(f'Alice bits: {alice_bits}')
print(f'Alice bases: {["Z" if b==0 else "X" for b in alice_bases]}')

Alice bits: [1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1]
Alice bases: ['X', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z']


In [15]:
# ── ALICE ──────────────────────────────────────────────────────────────────
def alice_prepare_qubits(bits, bases):
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits


qubit_circuits = alice_prepare_qubits(alice_bits, alice_bases)

In [16]:
# ── BOB ────────────────────────────────────────────────────────────────────
def bob_measure_qubit(qubit_circuit, basis):
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)        # Rotate back from X-basis before measuring
    qc.measure(0, 0)

    simulator = BasicSimulator()
    job       = simulator.run(transpile(qc, simulator), shots=1)
    counts    = job.result().get_counts()
    return int(list(counts.keys())[0])


bob_bases   = quantum_random_bits(N)   # Bob's random basis choices
bob_results = [
    bob_measure_qubit(qc, basis)
    for qc, basis in zip(qubit_circuits, bob_bases)
]

print(f'Bob bases   : {["Z" if b==0 else "X" for b in bob_bases]}')
print(f'Bob results : {bob_results}')

Bob bases   : ['X', 'X', 'X', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'X', 'X', 'X']
Bob results : [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1]


In [17]:
# ── ALICE + BOB (public classical channel) ─────────────────────────────────
alice_key, bob_key = [], []
for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
    if ab == bb:                      # Same basis -> reliable bit
        alice_key.append(alice_bits[i])
        bob_key.append(bob_results[i])

matching = [i for i, (a, b) in enumerate(zip(alice_bases, bob_bases)) if a == b]
print(f'Matching positions : {matching}  ({len(matching)} of {N})')
print(f'Alice sifted key   : {alice_key}')
print(f'Bob   sifted key   : {bob_key}')

Matching positions : [0, 1, 3, 5, 6, 8, 9, 11, 12, 13, 14, 17]  (12 of 20)
Alice sifted key   : [1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1]
Bob   sifted key   : [1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1]


In [18]:
# ── ALICE + BOB (public classical channel) ─────────────────────────────────

# Detection threshold: QBER > 11% -> attack likely
# Intercept-and-resend gives ~25% QBER; 11% sits safely between 0% (clean) and 25% (attacked)
ATTACK_THRESHOLD = 0.11

n       = len(alice_key)
n_check = n // 2                      # Sacrifice first half for error checking

check_alice = alice_key[:n_check]
check_bob   = bob_key[:n_check]
alice_final = alice_key[n_check:]     # Remaining bits become the secret key
bob_final   = bob_key[n_check:]

errors     = sum(a != b for a, b in zip(check_alice, check_bob))
error_rate = errors / n_check if n_check > 0 else 0.0

print(f'Bits used for error check : {n_check}')
print(f'QBER (error rate)         : {error_rate:.1%}')
print(f'Detection threshold       : {ATTACK_THRESHOLD:.1%}')

Bits used for error check : 6
QBER (error rate)         : 0.0%
Detection threshold       : 11.0%


In [19]:
# ── ALICE + BOB ────────────────────────────────────────────────────────────
attack_detected = error_rate > ATTACK_THRESHOLD

sep = '=' * 60
print(f'\n{sep}')
print('  BB84 Simulation — No Attacker')
print(sep)

print('\n--- Raw transmission (20 qubits) ---')
print(f'  Alice bits  : {alice_bits}')
print(f'  Alice bases : {["Z" if b==0 else "X" for b in alice_bases]}')
print(f'  Bob   bases : {["Z" if b==0 else "X" for b in bob_bases]}')
print(f'  Bob results : {bob_results}')

print(f'\n--- Basis sifting ---')
print(f'  Matching positions : {matching}  ({len(matching)} of {N})')
print(f'  Alice sifted key   : {alice_key}')
print(f'  Bob   sifted key   : {bob_key}')

print(f'\n--- Error checking (first {n_check} sifted bits sacrificed) ---')
print(f'  QBER (error rate)  : {error_rate:.1%}')
print(f'  Threshold          : {ATTACK_THRESHOLD:.1%}')

print(f'\n--- Attack detection ---')
print(f'  Attack detected    : {"YES" if attack_detected else "NO"}')

print(f'\n--- Final secret key (remaining {len(alice_final)} bits) ---')
if attack_detected:
    print('  Key discarded due to suspected eavesdropping.')
else:
    print(f'  Alice final key : {alice_final}')
    print(f'  Bob   final key : {bob_final}')
    print(f'  Keys match      : {"Yes" if alice_final == bob_final else "No"}')
print(sep)


  BB84 Simulation — No Attacker

--- Raw transmission (20 qubits) ---
  Alice bits  : [1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1]
  Alice bases : ['X', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z', 'Z', 'Z', 'Z', 'X', 'Z', 'X', 'X', 'Z', 'X', 'Z', 'Z']
  Bob   bases : ['X', 'X', 'X', 'X', 'Z', 'Z', 'X', 'X', 'Z', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'X', 'X', 'X']
  Bob results : [1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1]

--- Basis sifting ---
  Matching positions : [0, 1, 3, 5, 6, 8, 9, 11, 12, 13, 14, 17]  (12 of 20)
  Alice sifted key   : [1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1]
  Bob   sifted key   : [1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1]

--- Error checking (first 6 sifted bits sacrificed) ---
  QBER (error rate)  : 0.0%
  Threshold          : 11.0%

--- Attack detection ---
  Attack detected    : NO

--- Final secret key (remaining 6 bits) ---
  Alice final key : [1, 1, 1, 0, 0, 1]
  Bob   final key : [1, 1, 1, 0, 0, 1]
  Keys match      : Yes
